In [11]:
import pandas as pd
import os
import re

In [13]:
def load_data(data_name, study_list):
    source_files = []
    for dp, dn, filenames in os.walk(os.path.join('../..','data', 'processed_data')):
        for f in filenames:
            if re.search(f"{data_name}.csv$", f):
                source_files.append(os.path.join(dp, f))
    
    print("Source files found:", source_files)  # Debugging line
    
    data_files = [file for file in source_files if any(study in file for study in study_list)]
    
    print("Filtered data files:", data_files)  # Debugging line
    
    return pd.concat([pd.read_csv(file) for file in data_files], ignore_index=True) if data_files else pd.DataFrame()


target_experiment_names = ["pilot_v1-0", "pilot_v1-1", "pilot_v1-2"]

d_game = load_data("games", target_experiment_names)
d_round = load_data("rounds", target_experiment_names)
d_chat = load_data("chats", target_experiment_names)
d_players = load_data("players", target_experiment_names)

Source files found: ['../../data/processed_data/pilot_v1-0/games.csv', '../../data/processed_data/pilot_v1-1/games.csv', '../../data/processed_data/demo_v1/games.csv', '../../data/processed_data/pilot_v1-2/games.csv']
Filtered data files: ['../../data/processed_data/pilot_v1-0/games.csv', '../../data/processed_data/pilot_v1-1/games.csv', '../../data/processed_data/pilot_v1-2/games.csv']
Source files found: ['../../data/processed_data/pilot_v1-0/rounds.csv', '../../data/processed_data/pilot_v1-1/rounds.csv', '../../data/processed_data/demo_v1/rounds.csv', '../../data/processed_data/pilot_v1-2/rounds.csv']
Filtered data files: ['../../data/processed_data/pilot_v1-0/rounds.csv', '../../data/processed_data/pilot_v1-1/rounds.csv', '../../data/processed_data/pilot_v1-2/rounds.csv']
Source files found: ['../../data/processed_data/pilot_v1-0/chats.csv', '../../data/processed_data/pilot_v1-1/chats.csv', '../../data/processed_data/demo_v1/chats.csv', '../../data/processed_data/pilot_v1-2/chats.c

In [14]:
d_chat_clean = d_chat.merge(d_round, how='left').merge(d_game, how='left')

In [16]:
def clean_text(text):
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with single space
    text = text.strip()  # Remove leading and trailing whitespace
    return text

# Clean the text and compute lengths
d_chat_clean['text'] = d_chat_clean['text'].apply(clean_text)
d_chat_clean['utt_length_chars'] = d_chat_clean['text'].str.len()
d_chat_clean['utt_length_words'] = d_chat_clean['text'].str.split().str.len()


chat_by_trial = d_chat_clean.groupby(['gameID', 'roundID','index', 'repNum', 'playerID','director', 'contextStructure']).agg({
    'text': lambda x: ', '.join(x),
    'utt_length_words': 'sum',
    'utt_length_chars': 'sum'
}).reset_index()

chat_by_trial.rename(columns={'utt_length_words': 'total_num_words', 'utt_length_chars': 'total_num_chars'}, inplace=True)
chat_by_trial['index'] = chat_by_trial.index

Now, get the similarity between each utterance and each other utterance via embeddings

In [17]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [18]:
# Two lists of sentencesn watches TV",

# Compute embeddings for both lists
embeddings = model.encode(chat_by_trial['text'])

# Compute cosine similarities
similarities = model.similarity(embeddings, embeddings)

In [19]:
similarities

tensor([[ 1.0000,  0.2870,  0.0686,  ...,  0.1191,  0.2393,  0.0952],
        [ 0.2870,  1.0000,  0.5560,  ...,  0.2313,  0.0656,  0.1496],
        [ 0.0686,  0.5560,  1.0000,  ...,  0.2357, -0.0257,  0.1863],
        ...,
        [ 0.1191,  0.2313,  0.2357,  ...,  1.0000,  0.2536,  0.1855],
        [ 0.2393,  0.0656, -0.0257,  ...,  0.2536,  1.0000,  0.2628],
        [ 0.0952,  0.1496,  0.1863,  ...,  0.1855,  0.2628,  1.0000]])

In [20]:
chat_by_trial.rename(columns = {"index":"index1",
                                "roundID":"roundID1",
                                "repNum":"repNum1"})

,gameID,roundID1,index1,repNum1,playerID,director,contextStructure,text,total_num_words,total_num_chars
0,01J1G1WHT4GZEZV213005AZ1MM,01J1G2J4SQ4RSZ8S0KJM61T8HQ,0,0,01J1G2AW8JR5Q9P9WX8DF1C7DP,t,comp-between,"shaped like a persons head with a bob, or a tr...",15,62
1,01J1G1WHT9T93BKEMYX1GA4C93,01J1G2JBR9C2PRS08YBFPW09W9,1,0,01J1G28YDBHJP9AHTMTEE3RA7E,t,comp-within,"it has spikes on the top, with a triangle in d...",14,61
2,01J1G1WHT9T93BKEMYX1GA4C93,01J1G2JBRAKRWR59S6ZBNAKJJP,2,0,01J1G20MXD2JXJMM31WQFG1M43,t,comp-within,spikes on top with wine glass bottom,7,36
3,01J1G1WHT9T93BKEMYX1GA4C93,01J1G2JBRAKRWR59S6ZBNAKJJP,3,0,01J1G28YDBHJP9AHTMTEE3RA7E,f,comp-within,what does it look like,5,22
4,01J1G1WHT9T93BKEMYX1GA4C93,01J1G2JBRB7XTM0H53505JYDG1,4,0,01J1G28YDBHJP9AHTMTEE3RA7E,t,comp-within,spikes on top with a thin tree like bottom,9,42
...,...,...,...,...,...,...,...,...,...,...
1545,01J1QNEE1QGJVSMNPAJZ8D88KJ,01J1QR66TNNKC157E1RR9X76YN,1545,3,01J1QR0NNX3XKTWMQVWJ9M6XEV,t,comp-between,missing bottom right,3,20
1546,01J1QNEE1QGJVSMNPAJZ8D88KJ,01J1QR66V0MVR8GBP8X69760Q7,1546,3,01J1QR2HT24AKE7JX92ZR1W6VW,t,comp-between,missing bottom right corner,4,27
1547,01J1QNEE1QGJVSMNPAJZ8D88KJ,01J1QR66VECE02VB0N5PJ8RCYT,1547,3,01J1QR2HT24AKE7JX92ZR1W6VW,t,comp-between,fish pointing down,3,18
1548,01J1QNEE1QGJVSMNPAJZ8D88KJ,01J1QR66VS21RSM6TFXKZY78QJ,1548,3,01J1QR0NNX3XKTWMQVWJ9M6XEV,t,comp-between,human,1,5


In [21]:
similarity_df = pd.DataFrame(similarities.numpy(), index=chat_by_trial.index, columns=chat_by_trial.index)
result = similarity_df.stack().reset_index()

result.columns = ['index1', 'index2', 'value']

# Remove duplicate pairs if the matrix is symmetric
result = result[result['index1'] < result['index2']]

word1 = chat_by_trial.rename(columns = {"index":"index1",
                                "roundID":"roundID1",
                                "repNum":"repNum1",
                                "playerID":"playerID1",
                                "text":"text1"})[["index1",  "text1", "roundID1", "repNum1", "playerID1"]]

word2 = chat_by_trial.rename(columns = {"index":"index2",
                                "roundID":"roundID2",
                                "repNum":"repNum2",
                                "playerID":"playerID2",
                                "text":"text2"})[["index2",  "text2", "roundID2", "repNum2", "playerID2"]]

similarity_df_merged = result.merge(word1, how = "left").merge(word2, how = "left")

similarity_df_merged.to_csv(os.path.join('../..','data', 'processed_data', "chat_pairwise_similarities.csv"))